# Persistent Chat with D1

Build a chatbot that remembers conversations across sessions using Cloudflare D1 (serverless SQLite).

**Prerequisites:**
- API token with: **Workers AI → Read**, **D1 → Edit**
- A D1 database (instructions below)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv("../.env")

assert os.getenv("CLOUDFLARE_ACCOUNT_ID")
assert os.getenv("CLOUDFLARE_API_TOKEN")

## Create a D1 database

Run this once in your terminal:

```bash
npx wrangler d1 create my-chat-history
```

Copy the database UUID from the output and set it below.

In [ ]:
D1_DATABASE_ID = "your-d1-database-id"  # Replace with your UUID

## Multi-turn conversation with memory

In [ ]:
from pydantic_ai import Agent
from pydantic_ai_cloudflare import D1MessageHistory

agent = Agent(
    "openai:@cf/meta/llama-3.3-70b-instruct-fp8-fast",
    system_prompt="You are a helpful assistant. Remember previous messages.",
)

history = D1MessageHistory(database_id=D1_DATABASE_ID)
session_id = "demo-session"

# Load any previous conversation
messages = await history.get_messages(session_id)
print(f"Loaded {len(messages)} previous messages")

In [ ]:
# Turn 1: Introduce yourself
result = await agent.run(
    "My name is Alice and I'm a data scientist at Cloudflare.",
    message_history=messages,
)
messages = result.all_messages()
print(f"Assistant: {result.output}")

In [ ]:
# Turn 2: Test memory
result = await agent.run(
    "What's my name and where do I work?",
    message_history=messages,
)
messages = result.all_messages()
print(f"Assistant: {result.output}")

In [ ]:
# Save conversation to D1
await history.save_messages(session_id, messages)
print(f"Saved {len(messages)} messages to D1")

# List all sessions
sessions = await history.list_sessions()
for s in sessions:
    print(f"  Session '{s['session_id']}': {s['message_count']} messages")

## Why D1?

- **Serverless SQLite** — no connection strings, no database server
- **5 GB free** — plenty for chat history
- **Time Travel** — restore to any point in the last 30 days
- **HTTP API** — works from anywhere, not just Cloudflare Workers
- **Automatic table creation** — `D1MessageHistory` creates its table on first use